<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Régression linéaire par station et par heure <br>
RMSE public : <br>
RMSE perso : 
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bibliothèques
</div>

In [1]:
from Donnees_et_RMSE import *
from sklearn.linear_model import LinearRegression

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Paramètres
</div>

In [2]:
# Nombre minimal de jours d'entrainement pour faire la régression, sinon on prédit que rien ne change
MIN_JOURS = 10

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Modèle de prévision
</div>

In [3]:
# train_modele sont les données avec lesquelles le modèle va s'entraîner
# train_test est le fichier dans lequel le modèle va récupérer les dates à prédire et dans lequel il va écrire ses prédictions
def modele_prevision(train_modele, test_modele):
    # Initialisation -------------------------------------------------------------------------------------------
    # On extrait tous les indexe station et commit_at_index sous forme de liste
    station_index = test_modele.index.get_level_values("station")
    commit_at_index = test_modele.index.get_level_values("commit_at")

    # Stations qui ont encore des données dans train_modele (certaines n'en ont plus après le découpage de validation)
    stations_connues = set(train_modele.index.get_level_values("station").unique())
    
    # Valeur utilisée pour les stations sans aucune donnée
    moyenne_globale = train_modele["bikes"].mean()
    
    # Tableau dans lequel on stocke les prédictions (une ligne par station du test, une colonne par horizon)
    predictions = np.zeros((len(test_modele), len(decalage)))
    
    
    
    # Prédictions ----------------------------------------------------------------------------------------------
    for i, (station, t0) in enumerate(zip(station_index, commit_at_index)):
        if i % 100 == 0:
            print(f"{i / len(test_modele) * 100:.1f} %")

        # Station sans aucune donnée : on utilise la moyenne globale
        if station not in stations_connues:
            predictions[i, :] = moyenne_globale
            continue
        
        # Données de cette station uniquement
        data_station = train_modele.xs(station, level="station").copy()
        
        # On va sélectionner uniquement les observations faites à la même heure et à la même minute que t0
        data_station_small = data_station[
            (data_station.index.hour == t0.hour)
            & (data_station.index.minute == t0.minute)
        ].copy()
        dates = data_station_small.index
        
        # Nombre de vélos à t0 et à t0 - 1h
        x_new = pd.DataFrame({
            "bikes_avant": [data_station["bikes"].get(t0 - pd.Timedelta(hours=1), np.nan)],
            "bikes_t0": [data_station["bikes"].get(t0, np.nan)]
        })
        
        # Si il y a moins que MIN_JOURS, on utilise la dernière valeur connue
        securite = x_new["bikes_t0"][0] if not np.isnan(x_new["bikes_t0"][0]) else data_station["bikes"].mean()
        
        # Un modèle par décalage horaire
        for j, (col, h) in enumerate(decalage.items()):
            # Mise en colonnes des heures (une ligne par jour)
            data_station_hour = pd.DataFrame({
                "bikes_avant": data_station["bikes"].reindex(dates - pd.Timedelta(hours=1)).to_numpy(),
                "bikes_t0": data_station_small["bikes"].to_numpy(),
                "bikes_cible": data_station["bikes"].reindex(dates + pd.Timedelta(minutes=h)).to_numpy()
            }, index=dates)
            
            # Suppression de valeurs manquantes
            data_station_hour = data_station_hour.dropna()
            
            # Construction de X et Y
            X = data_station_hour[["bikes_avant", "bikes_t0"]]
            Y = data_station_hour["bikes_cible"]
            
            if len(data_station_hour) >= MIN_JOURS and not x_new.isna().any().any():
                # Si on a assez de données on fait une régression linéaire
                model = LinearRegression()
                model.fit(X, Y)
                prediction = model.predict(x_new)[0]
            else:
                prediction = securite
            
            # On empêche les prédictions d'être négatives ou de dépasser le maximum observé dans la station
            predictions[i, j] = np.clip(prediction, 0, data_station["bikes"].max())
    
    # On écrit les prédictions dans test_modele
    for j, col in enumerate(decalage):
        test_modele[col] = predictions[:, j].round().astype(int)
    
    # On renvoie les prédictions
    return test_modele

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
On entraine sur l'ensemble de validation et on fait une prédiction sur les dernières valeurs du train pour calculer le score RMSE perso <br>
Puis on ré-entraine sur TOUTES les données fournies pour faires les vraies prédictions <br>
Enfin on sauvegarde le résultat dans un fichier parquet
</div>

In [4]:
rmse, history_toulouse_prediction = RMSE_et_predictions(modele_prevision, "BA_prediction_100_bis_RL")
print(f"RMSE : {rmse:.4f}")
history_toulouse_prediction

0.0 %
12.5 %
25.0 %
37.5 %
50.0 %
62.5 %
75.0 %
87.5 %
0.0 %
12.5 %
25.0 %
37.5 %
50.0 %
62.5 %
75.0 %
87.5 %
RMSE : 2.1115


,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,9,8,7,7,6,6,5,5
00002 - LAFAYETTE,2026-07-31 23:45:00,9,9,8,8,8,8,8,8
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,17,16,16,16,15,14,13,13
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,11,10,9,8,7,6,4,2
00003 - POMME,2025-01-05 05:45:00,17,17,17,17,17,17,17,17
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
